<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/Firewall_Log_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To develop a Python program that analyzes simulated firewall logs containing allowed and blocked connections, identifies source IP addresses responsible for repeated blocked requests, determines targeted destination ports, and generates statistics highlighting repeated unauthorized-access attempts.

**Algorithm**

Create or load simulated firewall records.

Read timestamp, source IP, destination IP, destination port, protocol, and action.

Filter records where the firewall action is BLOCKED.

Group blocked requests by source IP address.

Count blocked requests generated by each source.

Identify the destination ports targeted by each source.

Calculate the most frequently blocked sources.

Calculate the most frequently targeted services/ports.

Apply a configurable threshold to identify repeated unauthorized attempts.

Display detailed evidence and final statistics.


In [1]:
# ==============================================
# FIREWALL LOG ANALYZER
# ==============================================

import pandas as pd

# ------------------------------------------------
# 1. Simulated Firewall Records
# ------------------------------------------------

data = [
    ["09:00:00", "192.168.1.50", "10.0.0.10", 22,   "TCP", "BLOCKED"],
    ["09:01:00", "192.168.1.50", "10.0.0.10", 22,   "TCP", "BLOCKED"],
    ["09:02:00", "192.168.1.50", "10.0.0.10", 22,   "TCP", "BLOCKED"],
    ["09:03:00", "192.168.1.50", "10.0.0.10", 3389, "TCP", "BLOCKED"],

    ["09:05:00", "192.168.1.60", "10.0.0.20", 80,   "TCP", "ALLOWED"],
    ["09:06:00", "192.168.1.60", "10.0.0.20", 443,  "TCP", "ALLOWED"],

    ["09:10:00", "10.10.10.5", "10.0.0.30", 23,   "TCP", "BLOCKED"],
    ["09:11:00", "10.10.10.5", "10.0.0.30", 23,   "TCP", "BLOCKED"],
    ["09:12:00", "10.10.10.5", "10.0.0.30", 23,   "TCP", "BLOCKED"],
    ["09:13:00", "10.10.10.5", "10.0.0.30", 22,   "TCP", "BLOCKED"],

    ["09:15:00", "172.16.0.5", "10.0.0.40", 445,  "TCP", "BLOCKED"],
    ["09:16:00", "172.16.0.5", "10.0.0.40", 445,  "TCP", "BLOCKED"],

    ["09:20:00", "192.168.1.70", "10.0.0.50", 443, "TCP", "ALLOWED"]
]

df = pd.DataFrame(
    data,
    columns=[
        "Time",
        "Source_IP",
        "Destination_IP",
        "Destination_Port",
        "Protocol",
        "Action"
    ]
)

print("=" * 95)
print("                     FIREWALL LOG ANALYZER")
print("=" * 95)

# ------------------------------------------------
# 2. Investigator Threshold
# ------------------------------------------------

threshold = int(
    input(
        "\nEnter blocked-attempt threshold: "
    )
)

# ------------------------------------------------
# 3. Filter Blocked Requests
# ------------------------------------------------

blocked = df[
    df["Action"] == "BLOCKED"
].copy()

print("\nTotal Blocked Requests:",
      len(blocked))

# ------------------------------------------------
# 4. Count Blocked Requests by Source
# ------------------------------------------------

source_stats = (
    blocked
    .groupby("Source_IP")
    .size()
    .reset_index(
        name="Blocked_Requests"
    )
    .sort_values(
        "Blocked_Requests",
        ascending=False
    )
)

# ------------------------------------------------
# 5. Identify Repeated Unauthorized Sources
# ------------------------------------------------

repeated = source_stats[
    source_stats["Blocked_Requests"] >= threshold
]

print("\n" + "=" * 95)
print("              REPEATED UNAUTHORIZED-ACCESS SOURCES")
print("=" * 95)

if repeated.empty:

    print(
        "No source crossed the selected threshold."
    )

else:

    print(
        repeated.to_string(index=False)
    )

# ------------------------------------------------
# 6. Destination Ports
# ------------------------------------------------

print("\n" + "=" * 95)
print("                  TARGETED PORTS")
print("=" * 95)

port_stats = (
    blocked
    .groupby("Destination_Port")
    .size()
    .reset_index(
        name="Blocked_Requests"
    )
    .sort_values(
        "Blocked_Requests",
        ascending=False
    )
)

print(
    port_stats.to_string(index=False)
)

# ------------------------------------------------
# 7. Common Service Names
# ------------------------------------------------

services = {
    21: "FTP",
    22: "SSH",
    23: "Telnet",
    25: "SMTP",
    53: "DNS",
    80: "HTTP",
    443: "HTTPS",
    445: "SMB",
    3389: "RDP"
}

port_stats["Service"] = (
    port_stats["Destination_Port"]
    .map(services)
    .fillna("Unknown")
)

print("\n" + "=" * 95)
print("                 MOST BLOCKED SERVICES")
print("=" * 95)

print(
    port_stats[
        [
            "Destination_Port",
            "Service",
            "Blocked_Requests"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------
# 8. Detailed Evidence for Repeated Sources
# ------------------------------------------------

print("\n" + "=" * 95)
print("                    SUPPORTING EVIDENCE")
print("=" * 95)

if repeated.empty:

    print("No repeated unauthorized attempts.")

else:

    for ip in repeated["Source_IP"]:

        records = blocked[
            blocked["Source_IP"] == ip
        ]

        print("\nSource IP:", ip)

        print(
            records[
                [
                    "Time",
                    "Destination_IP",
                    "Destination_Port",
                    "Protocol",
                    "Action"
                ]
            ].to_string(index=False)
        )

# ------------------------------------------------
# 9. Final Statistics
# ------------------------------------------------

print("\n" + "=" * 95)
print("                     FINAL STATISTICS")
print("=" * 95)

print(
    "Total Firewall Events     :",
    len(df)
)

print(
    "Blocked Requests          :",
    len(blocked)
)

print(
    "Allowed Requests          :",
    len(df[df["Action"] == "ALLOWED"])
)

print(
    "Repeated Blocked Sources  :",
    len(repeated)
)

print(
    "Most Blocked Source       :",
    source_stats.iloc[0]["Source_IP"]
)

print(
    "Most Targeted Port        :",
    port_stats.iloc[0]["Destination_Port"]
)

print("\nFirewall analysis completed.")
print("=" * 95)

                     FIREWALL LOG ANALYZER

Enter blocked-attempt threshold: 3

Total Blocked Requests: 10

              REPEATED UNAUTHORIZED-ACCESS SOURCES
   Source_IP  Blocked_Requests
  10.10.10.5                 4
192.168.1.50                 4

                  TARGETED PORTS
 Destination_Port  Blocked_Requests
               22                 4
               23                 3
              445                 2
             3389                 1

                 MOST BLOCKED SERVICES
 Destination_Port Service  Blocked_Requests
               22     SSH                 4
               23  Telnet                 3
              445     SMB                 2
             3389     RDP                 1

                    SUPPORTING EVIDENCE

Source IP: 10.10.10.5
    Time Destination_IP  Destination_Port Protocol  Action
09:10:00      10.0.0.30                23      TCP BLOCKED
09:11:00      10.0.0.30                23      TCP BLOCKED
09:12:00      10.0.0.30          

**Result**

The Python program successfully analyzed simulated firewall records and separated blocked and allowed connections. It identified repeated blocked requests from source addresses, determined the destination ports targeted, and generated statistics for the most frequently blocked sources and services. With a threshold of 3, 192.168.1.50 and 10.10.10.5 were identified as sources making repeated unauthorized-access attempts, with SSH (port 22) being the most frequently targeted service.